In [ ]:
# ========= CONFIG (fill locally; do NOT commit secrets) =========
# Tip: use environment variables or a local `.env` file.
# Example:
#   import os
#   OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
#   HF_TOKEN = os.getenv("HF_TOKEN", "")
# ===============================================================



# Enabling the GPU

First, you'll need to enable GPUs for the notebook:

- Navigate to Edit→Notebook Settings
- select GPU from the Hardware Accelerator drop-down

[Reference](https://colab.research.google.com/notebooks/gpu.ipynb)

# **Installing BERTopic**

We start by installing BERTopic from PyPi:

In [1]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

## Restart the Notebook
After installing BERTopic, some packages that were already loaded were updated and in order to correctly use them, we should now restart the notebook.

From the Menu:

Runtime → Restart Runtime

# Unfiltered Data
The first dataset we are processing is the unfiltered data, where the dataset includes patents not from the industry. This first step is to classify the patents by topic modelling and filter out those which do not belong to the industry

In [2]:
import pandas as pd
from google.colab import drive
import os


# Parameters for dataset
sector = 'mobile_phone'
dataset = 'topic_unfiltered'

# Navigate the folder location
drive.mount("/content/drive")
folder_path = f"/content/drive/MyDrive/TechShiftProject/{sector}/{dataset}_data/"


# Load the csv file
df = pd.read_csv(f"{folder_path}df_id_year_document.csv")
# Convert only float entries in 'document' column to strings
df['document'] = df['document'].apply(lambda x: str(x) if isinstance(x, float) else x)
df.to_csv(f"{folder_path}df_id_year_document.csv", index=False)

# Print the overall number of rows
print(f"Total number of rows: {len(df)}")

# Count the number of rows per unique year
year_counts = df['year'].value_counts().sort_index()

# Print the results to determine the start year and end yaer
for year, count in year_counts.items():
    print(f"Year: {year}, Count: {count}")

Mounted at /content/drive
Total number of rows: 811435
Year: 1901, Count: 1
Year: 1902, Count: 2
Year: 1903, Count: 3
Year: 1904, Count: 1
Year: 1905, Count: 2
Year: 1906, Count: 3
Year: 1907, Count: 6
Year: 1908, Count: 2
Year: 1909, Count: 3
Year: 1910, Count: 2
Year: 1911, Count: 2
Year: 1912, Count: 5
Year: 1913, Count: 1
Year: 1914, Count: 4
Year: 1915, Count: 3
Year: 1916, Count: 1
Year: 1919, Count: 3
Year: 1920, Count: 7
Year: 1921, Count: 8
Year: 1922, Count: 6
Year: 1923, Count: 5
Year: 1924, Count: 2
Year: 1925, Count: 1
Year: 1926, Count: 9
Year: 1927, Count: 8
Year: 1928, Count: 22
Year: 1929, Count: 14
Year: 1930, Count: 21
Year: 1931, Count: 31
Year: 1932, Count: 32
Year: 1933, Count: 19
Year: 1934, Count: 27
Year: 1935, Count: 22
Year: 1936, Count: 20
Year: 1937, Count: 32
Year: 1938, Count: 44
Year: 1939, Count: 29
Year: 1940, Count: 39
Year: 1941, Count: 20
Year: 1942, Count: 14
Year: 1943, Count: 9
Year: 1944, Count: 4
Year: 1945, Count: 8
Year: 1946, Count: 4
Year: 

In [4]:
# Choose the start year and end year
start_year = 1900
end_year = 2024    # the year to end

# Filter out all rows earlier than start_year
df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]
df.reset_index(drop=True, inplace=True)

# Extract contents to train on and corresponding ids
contents = df["document"]
ids = df["id"]

# Display the first content
print(len(contents))
print(contents[0])

811435
Mobile telephone location system mobile telephone location system provided wherein plurality mobile telephone exchanges controls area position monitor presence mobile telephone units within area connected national center national center capable receiving requests location mobile telephone units mobile telephone exchanges transmit location information via satellite mobile telephone exchanges within system mobile telephone exchange whose area paged mobile telephone unit located sends location information national center directly originating mobile telephone exchange location information contains information pertaining identity mobile telephone exchange mobile telephone data transmitted national center mobile telephone exchange issued request enabling exchange make connection


In [5]:
# Choose the size of minimum cluster
min_cluster_size = 300    # The larger the cluster size, the fewer the topics

# Prepare folders for further analysis
if dataset not in ("sampled", "test"):
    embedding_path = f'{folder_path}embedding_analysis_end_{end_year}/'
    bertopic_path = f"{folder_path}/bertopic_model_saved_{min_cluster_size}_end_{end_year}/"
else:
    embedding_path = f'{folder_path}embedding_analysis_{constant_patent_num}_{i_sampling}_end_{end_year}/'
    bertopic_path = f"{folder_path}/bertopic_model_saved_{min_cluster_size}_{constant_patent_num}_{i_sampling}_end_{end_year}/"

if not os.path.exists(embedding_path):
    os.makedirs(embedding_path)
if not os.path.exists(bertopic_path):
    os.makedirs(bertopic_path)

## **Pre-calculate Embeddings**
After having created our data, namely `contents`, we can dive into the very first best practice, **pre-calculating embeddings**.

BERTopic works by converting documents into numerical values, called embeddings. This process can be very costly, especially if we want to iterate over parameters. Instead, we can calculate those embeddings once and feed them to BERTopic to skip calculating embeddings each time.

In [6]:
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(contents, show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/25358 [00:00<?, ?it/s]

Save embeddings as a pickle file for later use so we don't have to train it again.

In [7]:
import pickle

# Assume your embeddings are calculated and stored in the 'embeddings' variable
# Save the embeddings
with open(f"{embedding_path}embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

Import the already saved embeddings

In [ ]:
import pickle

# Load the embeddings
with open(f"{embedding_path}/embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

## **Preventing Stochastic Behavior**
In BERTopic, we generally use a dimensionality reduction algorithm to reduce the size of the embeddings. This is done to prevent the [curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality) to a certain degree.

As a default, this is done with [UMAP](https://github.com/lmcinnes/umap) which is an incredible algorithm for reducing dimensional space. However, by default, it shows stochastic behavior which creates different results each time you run it. To prevent that, we will need to set a `random_state` of the model before passing it to BERTopic.

As a result, we can now fully reproduce the results each time we run the model.

In [8]:
from umap import UMAP

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

## **Controlling Number of Topics**
There is a parameter to control the number of topics, namely `nr_topics`. This parameter, however, merges topics **after** they have been created. It is a parameter that supports creating a fixed number of topics.

However, it is advised to control the number of topics through the cluster model which is by default HDBSCAN. HDBSCAN has a parameter, namely `min_topic_size` that indirectly controls the number of topics that will be created.

A higher `min_topic_size` will generate fewer topics and a lower `min_topic_size` will generate more topics.

Here, we will go with `min_topic_size=40` to get around XXX topics.

In [9]:
from hdbscan import HDBSCAN
hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', cluster_selection_method='eom', prediction_data=True)#, core_dist_n_jobs=1)

## **Improving Default Representation**
The default representation of topics is calculated through [c-TF-IDF](https://maartengr.github.io/BERTopic/algorithm/algorithm.html#5-topic-representation). However, c-TF-IDF is powered by the [CountVectorizer](https://maartengr.github.io/BERTopic/getting_started/vectorizers/vectorizers.html) which converts text into tokens. Using the CountVectorizer, we can do a number of things:

* Remove stopwords
* Ignore infrequent words
* Increase

In other words, we can preprocess the topic representations **after** documents are assigned to topics. This will not influence the clustering process in any way.

Here, we will ignore English stopwords and infrequent words. Moreover, by increasing the n-gram range we will consider topic representations that are made up of one or two words.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
stop_words = text.ENGLISH_STOP_WORDS.union(['first', 'second', 'third', 'sb', 'thereof', 'include',
                                            'method', 'including', 'includes', 'solve', 'solved', 'solution',
                                            'jpo', 'inpit', 'ncip', 'copyright', 'problem', 'invention',
                                            'innovation', 'provide', 'provides', 'provided', 'following',
                                            'result', 'describe', 'wherein', 'left', 'right', 'purpose', 'constitution',
                                            'january', 'jan', 'february', 'feb', 'march', 'mar', 'april', 'may', 'june',
                                            'july', 'august', 'aug', 'september', 'sep', 'october', 'oct', 'november', 'nov',
                                            'december', 'dec', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
                                            'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten',
                                            'eleven', 'twelve', 'thirteen', 'fourteen', 'fifteen', 'sixteen', 'seventeen',
                                            'eighteen', 'nineteen', 'twenty', 'thirty', 'forty', 'fifty', 'sixty',
                                            'seventy', 'eighty', 'ninety', 'hundred'])
stop_words = list(stop_words)
vectorizer_model = CountVectorizer(stop_words=stop_words, min_df=2, ngram_range=(1, 2))

## **Additional Representations**
Previously, we have tuned the default representation but there are quite a number of [other topic representations](https://maartengr.github.io/BERTopic/getting_started/representation/representation.html) in BERTopic that we can choose from. From [KeyBERTInspired](https://maartengr.github.io/BERTopic/getting_started/representation/representation.html#keybertinspired) and [PartOfSpeech](https://maartengr.github.io/BERTopic/getting_started/representation/representation.html#partofspeech), to [OpenAI's ChatGPT](https://maartengr.github.io/BERTopic/getting_started/representation/llm.html#chatgpt) and [open-source](https://maartengr.github.io/BERTopic/getting_started/representation/llm.html#langchain) alternatives, many representations are possible.

In BERTopic, you can model many different topic representations simultanously to test them out and get different perspectives of topic descriptions. This is called [multi-aspect](https://maartengr.github.io/BERTopic/getting_started/multiaspect/multiaspect.html) topic modeling.

Here, we will demonstrate a number of interesting and useful representations in BERTopic:

* KeyBERTInspired
  * A method that derives inspiration from how KeyBERT works
* PartOfSpeech
  * Using SpaCy's POS tagging to extract words
* MaximalMarginalRelevance
  * Diversify the topic words
* OpenAI
  * Use ChatGPT to label our topics


In [11]:
# import openai
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech

# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
pos_model = PartOfSpeech("en_core_web_sm")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# GPT-3.5
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
topic: <topic label>
"""
# client = openai.OpenAI(api_key=""  # TODO: fill in (do not commit))
# openai_model = OpenAI(client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    # "OpenAI": openai_model,  # Uncomment if you will use OpenAI
    "MMR": mmr_model,
    "POS": pos_model
}

## **Training**
Now that we have a set of best practices, we can use them in our training loop. Here, several different representations, keywords and labels for our topics will be created. If you want to iterate over the topic model it is advised to use the pre-calculated embeddings as that significantly speeds up training.

In [12]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
from bertopic import BERTopic

topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True,
  calculate_probabilities=False,
  low_memory=True
)

topics, probs = topic_model.fit_transform(contents, embeddings)


2025-05-04 13:37:06,542 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-04 14:02:24,272 - BERTopic - Dimensionality - Completed ✓
2025-05-04 14:02:24,289 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-04 14:06:54,523 - BERTopic - Cluster - Completed ✓
2025-05-04 14:06:54,666 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-05-04 14:10:53,237 - BERTopic - Representation - Completed ✓


In [13]:
# Save the model instantly after the training
topic_model.save(bertopic_path, serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

Load the model if needed

In [ ]:
from sentence_transformers import SentenceTransformer

# Define embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load model and add embedding model
topic_model = BERTopic.load(bertopic_path, embedding_model=embedding_model)

Save the output resuts to a dataframe

In [14]:
# Save the results of topic modelling to csv
df["topic"] = topics
# probs = [[round(prob, 4) for prob in sublist] for sublist in probs]
df["probs"] = probs    # round probs to 4 decimal places


# Save a year id topic csv for later use
df_id_date_topic = df.drop(columns=['document', 'probs'])
df_id_date_topic.to_csv(f'{bertopic_path}df_id_year_topic.csv', index=False)

# Get topic infomation
df_topic_info = topic_model.get_topic_info()
print(f'Number of topics: {len(df_topic_info)}')
df_topic_info.to_csv(f'{bertopic_path}topic_info.csv', index=False)

# Save the full dataframe excluding documnets to the csv file
df = df.drop(columns=['document'])
df.to_csv(f'{bertopic_path}topic_probs_matrix.csv', index=False)

Number of topics: 142


# Use topic info for sector identification

First compile the keywords of each topic

In [15]:
import ast

# Import topic info document
df_topic_info = pd.read_csv(f'{bertopic_path}topic_info.csv')

# Columns to convert from string to actual lists
keyword_columns = ""  # TODO: fill in (do not commit)

# Convert each column from string to list using ast.literal_eval
for col in keyword_columns:
    df_topic_info[col] = df_topic_info[col].apply(ast.literal_eval)

# Create a new 'keywords' column with the union of all keyword lists
df_topic_info['keywords'] = df_topic_info.apply(
    lambda row: list(set(row['Representation']) | set(row['KeyBERT']) | set(row['MMR']) | set(row['POS'])),
    axis=1
)

**Access OpenAI APIs to identify whether each topic is a sector-related technology**

Initiate OpenAI APIs

In [16]:
from openai import OpenAI
import pandas as pd
import time

# Initialize OpenAI client
client = OpenAI(api_key=""  # TODO: fill in (do not commit))


# Function to classify if a topic is {sector}-related
def classify_keywords_by_sector(keywords_list):
    # Construct the prompt
    prompt = (
        "Given the following list of keywords representing a technology topic:\n\n"
        f"{keywords_list}\n\n"
        f"Decide if this topic is significantly related to {sector} technology.\n\n"
        f"Include 'yes' if it clearly or frequently applies to {sector}, is specialized for {sector}, "
        f"or otherwise strongly supports {sector}-related development or use cases "
        f"(e.g., specialized hardware/software specifically tailored for {sector}, "
        f"advanced solutions critical for {sector} functionality, etc.).\n\n"
        "Say 'no' if the connection is extremely general-purpose or peripheral, "
        f"meaning it lacks direct or strong relevance to {sector} "
        f"(e.g., broad technologies with only potential indirect use in {sector}).\n\n"
        "Just respond with 'yes' or 'no'."
    )


    try:
        # Send request to GPT-4o-mini
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        # Extract and clean the response
        answer = response.choices[0].message.content.strip().lower()
        if 'yes' in answer:
            return 'yes'
        elif 'no' in answer:
            return 'no'
        else:
            return answer+'(not standard)'

    except Exception as e:
        print(f"API error: {e}")
        return 'error'



Enable API-identification for keywords

In [17]:
# Apply the classification row by row
sector_flags = []
for idx, row in df_topic_info.iterrows():
    result = classify_keywords_by_sector(row['keywords'])
    sector_flags.append(result)
    print(f"Row {idx} classified as: {result}")
    time.sleep(1.2)  # Respect API rate limits (adjust as needed)

# Add result to the dataframe
df_topic_info['keywords_is_sector_related'] = sector_flags

# Overwrite the original topic info
df_topic_info.to_csv(f'{bertopic_path}topic_info.csv', index=False)

Row 0 classified as: yes
Row 1 classified as: yes
Row 2 classified as: yes
Row 3 classified as: yes
Row 4 classified as: yes
Row 5 classified as: yes
Row 6 classified as: yes
Row 7 classified as: yes
Row 8 classified as: yes
Row 9 classified as: yes
Row 10 classified as: yes
Row 11 classified as: yes
Row 12 classified as: yes
Row 13 classified as: no
Row 14 classified as: yes
Row 15 classified as: yes
Row 16 classified as: yes
Row 17 classified as: yes
Row 18 classified as: yes
Row 19 classified as: yes
Row 20 classified as: yes
Row 21 classified as: yes
Row 22 classified as: no
Row 23 classified as: yes
Row 24 classified as: no
Row 25 classified as: yes
Row 26 classified as: no
Row 27 classified as: yes
Row 28 classified as: yes
Row 29 classified as: no
Row 30 classified as: no
Row 31 classified as: yes
Row 32 classified as: yes
Row 33 classified as: yes
Row 34 classified as: yes
Row 35 classified as: yes
Row 36 classified as: yes
Row 37 classified as: yes
Row 38 classified as: yes
Ro

Then get 3 representative columns of df_topic_info

In [18]:
# Access df_topic_info
df_topic_info = pd.read_csv(f'{bertopic_path}topic_info.csv')

import ast

# Convert each row's string into a Python list
df_topic_info['docs_list'] = df_topic_info['Representative_Docs'].apply(ast.literal_eval)

# Extract doc1, doc2, doc3 into separate columns (handle edge cases if needed)
df_topic_info['doc1'] = df_topic_info['docs_list'].apply(lambda lst: lst[0] if len(lst) > 0 else "")
df_topic_info['doc2'] = df_topic_info['docs_list'].apply(lambda lst: lst[1] if len(lst) > 1 else "")
df_topic_info['doc3'] = df_topic_info['docs_list'].apply(lambda lst: lst[2] if len(lst) > 2 else "")

df_topic_info

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs,keywords,keywords_is_sector_related,docs_list,doc1,doc2,doc3
0,-1,390147,-1_device_mobile_information_terminal,"['device', 'mobile', 'information', 'terminal'...","['mobile terminal', 'mobile communication', 'm...","['mobile', 'information', 'communication', 'wi...","['device', 'mobile', 'information', 'terminal'...",['Information protection method of mobile comm...,"['signal', 'display', 'electronic device', 'se...",yes,[Information protection method of mobile commu...,Information protection method of mobile commun...,Display device and method of controlling there...,"Display device, multi-display device, and imag..."
1,0,31018,0_semiconductor_film_semiconductor device_layer,"['semiconductor', 'film', 'semiconductor devic...","['manufacturing semiconductor', 'device semico...","['semiconductor', 'semiconductor device', 'sil...","['semiconductor', 'film', 'layer', 'substrate'...",['Substrate accommodating apparatus and method...,"['substrate', 'etching', 'gate electrode', 'in...",yes,[Substrate accommodating apparatus and method ...,Substrate accommodating apparatus and method f...,Semiconductor device and manufacturing method ...,Method for manufacturing semiconductor device ...
2,1,30101,1_image_camera_lens_shooting,"['image', 'camera', 'lens', 'shooting', 'recog...","['camera module', 'image sensor', 'image proce...","['camera', 'lens', 'recognition', 'mobile term...","['image', 'camera', 'lens', 'shooting', 'recog...",['Method and apparatus for configuring a relay...,"['lens', 'shooting', 'image forming', 'imaging...",yes,[Method and apparatus for configuring a relay ...,Method and apparatus for configuring a relay n...,Communication method and mobile terminal prese...,A kind of operating method and mobile terminal...
3,2,21163,2_charging_battery_wireless charging_battery cell,"['charging', 'battery', 'wireless charging', '...","['charging device', 'charging control', 'wirel...","['charging', 'wireless charging', 'battery cel...","['charging', 'battery', 'voltage', 'charge', '...",['Method and mobile device for migrating finan...,"['device charging', 'module', 'charging', 'cur...",yes,[Method and mobile device for migrating financ...,Method and mobile device for migrating finance...,Method and device for safely operating file up...,Method and system for controlling mobile termi...
4,3,16983,3_memory_memory cell_memory device_semiconduct...,"['memory', 'memory cell', 'memory device', 'se...","['semiconductor memory', 'memory device', 'mem...","['memory cell', 'memory device', 'semiconducto...","['memory', 'semiconductor', 'cell', 'line', 'b...",['Nonaqueous electrolyte cell semiconductor in...,"['line', 'semiconductor', 'cell', 'transistor'...",yes,[Nonaqueous electrolyte cell semiconductor int...,Nonaqueous electrolyte cell semiconductor inte...,Semiconductor device and manufacturing method ...,Semiconductor memory device semiconductor memo...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,136,316,136_beam failure_beam_failure_failure recovery,"['beam failure', 'beam', 'failure', 'failure r...","['beam recovery', 'recovery wireless', 'failur...","['beam failure', 'failure recovery', 'bfr', 'f...","['beam', 'failure', 'recovery', 'bfr', 'resour...",['Beam failure recovery method and apparatus a...,"['recovery', 'failure recovery', 'recovering b...",yes,[Beam failure recovery method and apparatus ap...,Beam failure recovery method and apparatus app...,Beam failure detection for wireless communicat...,Beam failure detection for wireless communicat...
138,137,314,137_congestion_congestion control_network_netw...,"['congestion', 'congestion control', 'network'...","['congestion control', 'controlling congestion...","['congestion', 'congestion control', 'network ...","['congestion', 'network', 'control', 'node', '...",['Congestion control in a wireless mobile syst...,"['indication', 'congestion indication', 'conge...",yes,[Congestion control in

Enable API-identification for representative documents

In [19]:
from openai import OpenAI
import pandas as pd
import time

# Initialize OpenAI client
client = OpenAI(api_key=""  # TODO: fill in (do not commit))

# Classification function for a *document*, not a list of keywords
def classify_document_by_sector(document_text):
    """
    Calls the GPT-4o-mini API to decide if a doc is {sector}-related.
    Returns 'yes' or 'no', or 'error' if the API fails, or fallback if response is unexpected.
    """
    prompt = (
        "Given the following text describing a technology topic:\n\n"
        f"{document_text}\n\n"
        f"Decide if this topic is significantly related to {sector} technology.\n\n"
        f"Include 'yes' if it clearly or frequently applies to {sector}, is specialized for {sector}, "
        f"or otherwise strongly supports {sector}-related development or use cases.\n\n"
        "Say 'no' if the connection is extremely general-purpose or peripheral, "
        f"meaning it lacks direct or strong relevance to {sector} "
        "(e.g., a broad or generic solution with only a potential indirect use).\n\n"
        "Just respond with 'yes' or 'no'."
    )
    try:
        # Send request to GPT-4o-mini
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )
        answer = response.choices[0].message.content.strip().lower()

        # Basic check for yes/no
        if 'yes' in answer:
            return 'yes'
        elif 'no' in answer:
            return 'no'
        else:
            return f"{answer}(not standard)"
    except Exception as e:
        print(f"API error: {e}")
        return 'error'

Enable API-identification for documents

In [20]:
# Classify each doc using the GPT-4o-mini function
doc1_sector = []
doc2_sector = []
doc3_sector = []

for idx, row in df_topic_info.iterrows():
    print(f"\nProcessing row {idx}...")

    r1 = classify_document_by_sector(row['doc1'])
    print(f"  doc1 classified as: {r1}")
    doc1_sector.append(r1)

    r2 = classify_document_by_sector(row['doc2'])
    print(f"  doc2 classified as: {r2}")
    doc2_sector.append(r2)

    r3 = classify_document_by_sector(row['doc3'])
    print(f"  doc3 classified as: {r3}")
    doc3_sector.append(r3)

# Store results in new columns
df_topic_info['doc1_sector_related'] = doc1_sector
df_topic_info['doc2_sector_related'] = doc2_sector
df_topic_info['doc3_sector_related'] = doc3_sector

# Save the updated DataFrame
df_topic_info.to_csv(f'{bertopic_path}topic_info.csv', index=False)

print("Classification complete!")


Processing row 0...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processing row 1...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processing row 2...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processing row 3...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processing row 4...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processing row 5...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processing row 6...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processing row 7...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processing row 8...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processing row 9...
  doc1 classified as: yes
  doc2 classified as: yes
  doc3 classified as: yes

Processin

Calculate the sector score by applying the given scoring rules to each of the four columns and summing the results

In [21]:
# Assign scores based on values in each column
df_topic_info['is_sector_score_total'] = (
    df_topic_info['keywords_is_sector_related'].apply(lambda x: 0.4 if x == 'yes' else 0) +
    df_topic_info['doc1_sector_related'].apply(lambda x: 0.2 if x == 'yes' else 0) +
    df_topic_info['doc2_sector_related'].apply(lambda x: 0.2 if x == 'yes' else 0) +
    df_topic_info['doc3_sector_related'].apply(lambda x: 0.2 if x == 'yes' else 0)
)

# Overwrite the csv file
df_topic_info.to_csv(f'{bertopic_path}topic_info.csv', index=False)

# Identify patents from the industry to be analyzed
Choose the patents from the topic with >0.5 'is_sector_score_total'

In [22]:
# Access df_topic_info
df_topic_info = pd.read_csv(f'{bertopic_path}topic_info.csv')

# Access patent topic correspondence
df_patent_topic = pd.read_csv(f'{bertopic_path}topic_probs_matrix.csv')

# Access the all patents
df_id_year_document = pd.read_csv(f"{folder_path}df_id_year_document.csv")


# Filter patents according to their topics' scores
# Step 1: Create the list of valid topics
valid_topics = df_topic_info[df_topic_info['is_sector_score_total'] > 0.5]['Topic'].tolist()

# Step 2: Merge df_id_year_document with df_patent_topic on 'id'
df_id_year_document = df_id_year_document.merge(df_patent_topic[['id', 'topic']], on='id', how='left')

# Step 3: Filter df_id_year_document to keep only rows where 'topic' is in valid_topics
df_id_year_document = df_id_year_document[df_id_year_document['topic'].isin(valid_topics)]

print(len(df_id_year_document))

# Save df_id_year_document
output_dataset = 'complete'
output_path = f"/content/drive/MyDrive/TechShiftProject/{sector}/{output_dataset}_data/"

df_id_year_document.to_csv(f'{output_path}df_id_year_document.csv', index=False)

# Filter df_company_id_year
df_company_id_year = pd.read_csv(f"{output_path}df_company_id_year.csv")
df_company_id_year = df_company_id_year[df_company_id_year['id'].isin(df_id_year_document['id'])]
df_company_id_year.to_csv(f'{output_path}df_company_id_year.csv', index=False)

752488
